# 24.07 - Submission for CV

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** CV submission checker notes.

**Priority:** P0 — avoid losing points to formatting mistakes.

Turn image/video model outputs into a submission that preserves test order, reverses the label mapping correctly, and passes strict structural checks.

## Core Ideas

- **Test order is data:** predictions must remain aligned with the supplied test IDs.
- **Invert the training mapping:** if training used `label -> index`, submission needs `index -> label`.
- **Validate before writing:** check column names, row count, duplicate/missing/unexpected IDs, label vocabulary, and NaN values.
- **Write `index=False`:** an accidental pandas index becomes an unwanted CSV column.
- **Reload the CSV:** the file on disk—not the in-memory DataFrame—is the actual deliverable.
- **Repair deliberately:** a fallback label can make a file structurally valid, but it should never hide a broken prediction pipeline.

## Setup and Prepared Predictions

The IDs are intentionally not sorted. Keep their exact order throughout the notebook. The logits represent three CV classes and need inverse label mapping.

In [1]:
import os
import numpy as np
import pandas as pd

TEST_IDS = np.array([
    "img_004.jpg",
    "img_001.jpg",
    "video_003.mp4",
    "img_002.jpg",
    "video_001.mp4",
    "video_002.mp4",
], dtype=object)

LOGITS = np.array([
    [3.2, 0.4, -0.5],
    [0.1, 2.8, 0.3],
    [-0.2, 0.7, 2.4],
    [0.6, 1.9, 0.2],
    [2.5, 0.3, 0.1],
    [0.4, 0.5, 1.8],
], dtype=np.float32)

INDEX_TO_LABEL = {0: "cat", 1: "dog", 2: "bird"}
ALLOWED_LABELS = ["cat", "dog", "bird"]

print("test IDs:", TEST_IDS.tolist())
print("logits shape:", LOGITS.shape)

test IDs: ['img_004.jpg', 'img_001.jpg', 'video_003.mp4', 'img_002.jpg', 'video_001.mp4', 'video_002.mp4']
logits shape: (6, 3)


## Exercise 24-A: Decode logits with the inverse label mapping

Choose the largest logit per row, then convert each class index back to the required label string. Reject non-finite logits and incomplete mappings.

**Return structure — `decode_logits(logits, index_to_label)`:**

- Returns a one-dimensional `numpy.ndarray` of shape `[N]` and dtype `object`.
- Every item is a label `str` obtained from `index_to_label`.
- Output row `i` corresponds to input logits row `i`; input order is unchanged.

In [3]:
# TODO 24-A
def decode_logits(logits, index_to_label):
    preds = logits.argmax(axis = 1)
    preds = np.array([index_to_label[i] for i in preds]).astype(object)
    return preds


# Smoke check: run this after implementing the function above.
smoke_labels = decode_logits(LOGITS, INDEX_TO_LABEL)
print("smoke labels:", smoke_labels.tolist())

smoke labels: ['cat', 'dog', 'bird', 'dog', 'cat', 'bird']


## Exercise 24-B: Create an ordered submission table

Combine IDs and labels without sorting, resetting from a different index, or adding extra columns.

**Return structure — `make_submission(test_ids, predicted_labels)`:**

- Returns a `pandas.DataFrame` with shape `[N, 2]`.
- Columns are exactly `['id', 'label']` in that order.
- `id` and `label` cells contain strings.
- Row `i` contains `test_ids[i]` and `predicted_labels[i]`.

In [4]:
# TODO 24-B
def make_submission(test_ids, predicted_labels):
    # Validate both one-dimensional inputs and preserve their order.
    if len(test_ids) != len(predicted_labels) : 
        raise ValueError()
    df = pd.DataFrame({
        "id" : test_ids.astype(str),
        "label" : predicted_labels.astype(str)
    })
    return df


# Smoke check: run this after implementing the function above.
smoke_submission = make_submission(TEST_IDS, smoke_labels)
print(smoke_submission)

              id label
0    img_004.jpg   cat
1    img_001.jpg   dog
2  video_003.mp4  bird
3    img_002.jpg   dog
4  video_001.mp4   cat
5  video_002.mp4  bird


## Exercise 24-C: Build a strict submission checker

Collect all structural problems in one report instead of stopping after the first error.

**Return structure — `check_submission(submission, expected_ids, allowed_labels)`:**

- Returns a dictionary with exactly nine keys:
  - `"valid"`: `bool`; `True` only when no error is found.
  - `"errors"`: `list[str]`; zero or more human-readable error messages.
  - `"row_count"`: `int`; actual DataFrame row count.
  - `"missing_ids"`: `list[str]` in expected-ID order.
  - `"unexpected_ids"`: sorted `list[str]`.
  - `"duplicate_ids"`: sorted `list[str]`.
  - `"invalid_labels"`: sorted `list[str]` excluding NaN values.
  - `"nan_cells"`: `int`; total missing cells.
  - `"order_matches"`: `bool`; whether the ID column exactly matches `expected_ids`.

In [5]:
# TODO 24-C
def check_submission(submission, expected_ids, allowed_labels):
    expected_ids = [str(value) for value in expected_ids]
    allowed_set = {str(value) for value in allowed_labels}
    errors = []
    row_count = int(len(submission))
    nan_cells = int(submission.isna().sum().sum())

    columns_match = submission.columns.tolist() == ["id", "label"]
    if not columns_match:
        errors.append("columns must be exactly ['id', 'label']")
    if row_count != len(expected_ids):
        errors.append(f"row count must be {len(expected_ids)}, got {row_count}")
    if nan_cells:
        errors.append(f"submission contains {nan_cells} NaN cell(s)")

    if "id" in submission.columns:
        current_ids = submission["id"].dropna().astype(str).tolist()
        duplicate_ids = sorted(set(submission.loc[submission["id"].duplicated(keep=False), "id"].dropna().astype(str)))
    else:
        current_ids = []
        duplicate_ids = []

    current_set = set(current_ids)
    expected_set = set(expected_ids)
    missing_ids = [value for value in expected_ids if value not in current_set]
    unexpected_ids = sorted(current_set - expected_set)
    order_matches = current_ids == expected_ids

    if duplicate_ids:
        errors.append(f"duplicate IDs: {duplicate_ids}")
    if missing_ids:
        errors.append(f"missing IDs: {missing_ids}")
    if unexpected_ids:
        errors.append(f"unexpected IDs: {unexpected_ids}")
    if not order_matches:
        errors.append("ID order does not match expected test order")

    if "label" in submission.columns:
        present_labels = submission["label"].dropna().astype(str)
        invalid_labels = sorted(set(present_labels) - allowed_set)
    else:
        invalid_labels = []
    if invalid_labels:
        errors.append(f"invalid labels: {invalid_labels}")

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "row_count": row_count,
        "missing_ids": missing_ids,
        "unexpected_ids": unexpected_ids,
        "duplicate_ids": duplicate_ids,
        "invalid_labels": invalid_labels,
        "nan_cells": nan_cells,
        "order_matches": order_matches,
    }


# Smoke check: run this after implementing the function above.
smoke_report = check_submission(smoke_submission, TEST_IDS, ALLOWED_LABELS)
print("smoke report:", smoke_report)

smoke report: {'valid': True, 'errors': [], 'row_count': 6, 'missing_ids': [], 'unexpected_ids': [], 'duplicate_ids': [], 'invalid_labels': [], 'nan_cells': 0, 'order_matches': True}


## Exercise 24-D: Repair an intentionally broken table

The prepared broken table has wrong order, a duplicate ID, an unexpected ID, missing IDs, a NaN label, and an invalid label. Reindex to the expected IDs and use a clearly declared fallback label when a usable prediction is absent.

**Return structure — `repair_submission(submission, expected_ids, allowed_labels, fallback_label)`:**

- Returns a `pandas.DataFrame` with shape `[N, 2]` and columns exactly `['id', 'label']`.
- Its `id` column exactly matches `expected_ids` in order.
- Every label is a non-missing string contained in `allowed_labels`.
- Duplicate IDs keep their first row; missing or invalid labels use `fallback_label`.
- The input DataFrame is not modified.

In [8]:
BROKEN_SUBMISSION = pd.DataFrame({
    "id": [TEST_IDS[1], TEST_IDS[0], TEST_IDS[0], "ghost.jpg", TEST_IDS[3]],
    "label": [smoke_labels[1], None, smoke_labels[0], "cat", "not-a-class"],
})

# TODO 24-D
def repair_submission(submission, expected_ids, allowed_labels, fallback_label):
    expected_ids = [str(value) for value in expected_ids]
    allowed_set = {str(value) for value in allowed_labels}
    fallback_label = str(fallback_label)
    if fallback_label not in allowed_set:
        raise ValueError("fallback_label must be in allowed_labels")

    work = submission.copy()
    if "id" not in work.columns:
        work["id"] = pd.Series(dtype=object)
    if "label" not in work.columns:
        work["label"] = pd.Series(dtype=object)

    work = work[["id", "label"]].dropna(subset=["id"])
    work["id"] = work["id"].astype(str)
    work = work[work["id"].isin(expected_ids)].drop_duplicates(subset="id", keep="first")
    label_by_id = work.set_index("id")["label"].to_dict()

    repaired_labels = []
    for test_id in expected_ids:
        label = label_by_id.get(test_id, fallback_label)
        if pd.isna(label) or str(label) not in allowed_set:
            label = fallback_label
        repaired_labels.append(str(label))

    return pd.DataFrame({"id": expected_ids, "label": repaired_labels}, columns=["id", "label"])


# Smoke check: run this after implementing the function above.
smoke_repaired = repair_submission(
    BROKEN_SUBMISSION,
    TEST_IDS,
    ALLOWED_LABELS,
    fallback_label="cat",
)
print(smoke_repaired)
print(check_submission(smoke_repaired, TEST_IDS, ALLOWED_LABELS))

              id label
0    img_004.jpg   cat
1    img_001.jpg   dog
2  video_003.mp4   cat
3    img_002.jpg   cat
4  video_001.mp4   cat
5  video_002.mp4   cat
{'valid': True, 'errors': [], 'row_count': 6, 'missing_ids': [], 'unexpected_ids': [], 'duplicate_ids': [], 'invalid_labels': [], 'nan_cells': 0, 'order_matches': True}


## Exercise 24-E: Save and reload the checked CSV

Only a valid table should reach disk. Save without a pandas index and reload it to verify the actual artifact.

**Return structure — `save_submission_csv(submission, output_path, expected_ids, allowed_labels)`:**

- Returns a `str` equal to `output_path`.
- Side effect: creates parent directories when needed and writes one UTF-8 CSV file.
- The file contains exactly two columns, `id` and `label`, and `N` data rows.
- Raises `ValueError` instead of writing when the checker reports an invalid submission.

In [6]:
# TODO 24-E
def save_submission_csv(submission, output_path, expected_ids, allowed_labels):
    report = check_submission(submission, expected_ids, allowed_labels)
    if not report["valid"]:
        raise ValueError("invalid submission: " + "; ".join(report["errors"]))

    parent = os.path.dirname(output_path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    submission.to_csv(output_path, index=False, encoding="utf-8")

    reloaded = pd.read_csv(output_path)
    disk_report = check_submission(reloaded, expected_ids, allowed_labels)
    if not disk_report["valid"]:
        raise ValueError("saved CSV failed validation: " + "; ".join(disk_report["errors"]))
    return output_path


# Smoke check: run this after implementing the function above.
smoke_path = save_submission_csv(
    smoke_submission,
    os.path.join("_day24_submission_output", "smoke_submission.csv"),
    TEST_IDS,
    ALLOWED_LABELS,
)
print("saved:", smoke_path)
print(pd.read_csv(smoke_path))

saved: _day24_submission_output\smoke_submission.csv
              id label
0    img_004.jpg   cat
1    img_001.jpg   dog
2  video_003.mp4  bird
3    img_002.jpg   dog
4  video_001.mp4   cat
5  video_002.mp4  bird


## Test Cases

Run this cell after completing all TODO cells.

**Return structure — `run_day24_tests()`:**

- Returns `None`.
- Success is communicated by completing all assertions and printing exactly `Day 24 tests passed`.

In [9]:
def run_day24_tests():
    assert "decode_logits" in globals(), "Missing function: decode_logits"
    assert "make_submission" in globals(), "Missing function: make_submission"
    assert "check_submission" in globals(), "Missing function: check_submission"
    assert "repair_submission" in globals(), "Missing function: repair_submission"
    assert "save_submission_csv" in globals(), "Missing function: save_submission_csv"

    labels = decode_logits(LOGITS, INDEX_TO_LABEL)
    assert labels.shape == (6,) and labels.dtype == object
    assert labels.tolist() == ["cat", "dog", "bird", "dog", "cat", "bird"]

    submission = make_submission(TEST_IDS, labels)
    assert submission.shape == (6, 2)
    assert submission.columns.tolist() == ["id", "label"]
    assert submission["id"].tolist() == TEST_IDS.tolist()

    valid_report = check_submission(submission, TEST_IDS, ALLOWED_LABELS)
    assert set(valid_report) == {
        "valid", "errors", "row_count", "missing_ids", "unexpected_ids",
        "duplicate_ids", "invalid_labels", "nan_cells", "order_matches",
    }
    assert valid_report["valid"] and valid_report["errors"] == []
    assert valid_report["row_count"] == len(TEST_IDS)
    assert valid_report["order_matches"]

    broken_report = check_submission(BROKEN_SUBMISSION, TEST_IDS, ALLOWED_LABELS)
    assert not broken_report["valid"]
    assert broken_report["duplicate_ids"] == ["img_004.jpg"]
    assert "ghost.jpg" in broken_report["unexpected_ids"]
    assert broken_report["nan_cells"] == 1
    assert broken_report["invalid_labels"] == ["not-a-class"]

    repaired = repair_submission(BROKEN_SUBMISSION, TEST_IDS, ALLOWED_LABELS, "cat")
    repaired_report = check_submission(repaired, TEST_IDS, ALLOWED_LABELS)
    assert repaired_report["valid"], repaired_report["errors"]
    assert repaired["id"].tolist() == TEST_IDS.tolist()

    output_path = os.path.join("_day24_submission_output", "test_submission.csv")
    returned_path = save_submission_csv(submission, output_path, TEST_IDS, ALLOWED_LABELS)
    assert returned_path == output_path and os.path.isfile(output_path)
    reloaded = pd.read_csv(output_path)
    assert reloaded.columns.tolist() == ["id", "label"]
    assert reloaded["id"].tolist() == TEST_IDS.tolist()
    assert reloaded["label"].tolist() == labels.tolist()

    print("Day 24 tests passed")


run_day24_tests()

Day 24 tests passed


## Day 24 Checklist

- [ ] I preserve the exact test-ID order from prediction through CSV.
- [ ] I use the inverse class mapping expected by the competition.
- [ ] I check row count, column order, duplicate/missing/unexpected IDs, NaN values, and label vocabulary.
- [ ] I write CSV files with `index=False`.
- [ ] I reload and validate the file from disk.
- [ ] I can intentionally break a submission and explain every checker error.
- [ ] I wrote my final CV submission checker notes.